# LakePilot Revision — Phase 1 multi-seed offline training (Kaggle GPU)

Trains **all specialist checkpoints for the Scientific Reports revision**:
3 architectures (AttentivePPO, MLP-PPO, DuelingDDQN) × 2 specialists (compaction, partition) × 5 seeds = **30 runs**.

**How to run**
1. Create a Kaggle dataset from `revision/phase1_stats/kaggle_training_data.zip` (heuristic transition CSVs) and attach it to this notebook.
2. Turn on a GPU accelerator (P100/T4 is fine). Expected total runtime ≈ 2–4 h.
3. Run all cells. Output: `/kaggle/working/phase1_weights.zip` containing
   `weights/{specialist}_{arch}_seed{k}.weights.h5` + one `manifest.json` per run + training curves.
4. Download the zip and unpack into `revision/phase1_stats/weights/` in the repo, then run the local
   evaluation batch (`revision_phase1_eval.py --policy MultiAgentRL ...`).

**Faithfulness to the published pipeline** (documented deviations only):
- Model classes are exact copies of `LakeGymLite/src/agents/model_registry.py` so weights load in the app.
  (The originally published offline DDQN notebook used a slightly different architecture without BatchNorm,
  whose weights could not be loaded by the application's model registry.)
- Fixed z-score constants for `rows_ingested`/`ingestion_rate` (mean 94/std 71, mean 318/std 218) —
  shared with deployment. The original notebooks z-scored per-dataset while deployment used (5,3)/(10,8),
  a train/serve skew fixed for the revision.
- DDQN trains on **raw dueling Q-values** with a proper Double-DQN target (the published online update
  regressed softmax outputs — defect documented in the response letter).
- DDQN replay transitions are paired **consecutively within episodes** (the original offline notebook
  paired shuffled train rows, making s′ a random unrelated state — defect fixed and documented).
- Per-episode CSV files only (`*_ep*.csv`) — avoids the duplicate combined+per-episode loading
  possible under the original glob patterns.
- Everything else replicates the published recipes **per architecture** (verified against the
  original notebooks, including their differing budgets and algorithms):
  - AttentivePPO compaction: AWR on discounted `compact_reward` returns, 150 epochs, batch 64.
  - AttentivePPO partition: AWR on the shaped reward (0.3·outcome + 0.5·alignment + 0.2·global),
    per-action standardized advantages, balanced action sampling, entropy 0.10, **200 epochs**, batch 64.
  - MLP-PPO (both specialists): **offline PPO-clip** (old-logp snapshotted from the initial network),
    150 epochs, batch 128, entropy 0.01, grad clip 0.5 — the published MLP recipe, which differs from
    AttentivePPO's in algorithm, budget, batch size, and (for partition) lack of balanced sampling.
  - DDQN (both specialists): Double-DQN, 150 epochs, batch 64, target-update every 50 steps.
    No original DDQN-*compaction* notebook exists in the repo (only its weights); its recipe is
    assumed to mirror the DDQN-partition notebook applied to `compact_reward` — flagged as an assumption.

  ⚠ These per-architecture differences (epochs 200 vs 150, AWR vs PPO-clip, batch 64 vs 128) mean the
  published architecture comparison is confounded by training recipe, not just capacity. Phase 1
  reproduces the published recipes faithfully; the fully recipe-matched controlled comparison
  (identical algorithm/budget, parameter-matched MLP) is Phase 7.


In [ ]:
# ── Cell 1: setup, constants, hardware ──
import os, glob, json, time, zipfile, platform, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

print("TF:", tf.__version__)
GPUS = tf.config.list_physical_devices('GPU')
print("GPUs:", GPUS)
try:
    GPU_NAME = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
except Exception:
    GPU_NAME = 'none'
print("GPU:", GPU_NAME)

SEEDS = [1, 2, 3, 4, 5]
ARCHS = ['attentive_ppo', 'mlp_ppo', 'ddqn']

OUTPUT_DIR = Path('/kaggle/working/phase1_weights') if Path('/kaggle').exists() else Path('./phase1_weights')
(OUTPUT_DIR / 'weights').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'curves').mkdir(parents=True, exist_ok=True)

# ── Fixed normalization constants (MUST match src/agents/compaction_agent.py) ──
ROWS_MEAN, ROWS_STD = 94.0, 71.0
RATE_MEAN, RATE_STD = 318.0, 218.0

# ── Reward constants (MUST match src/simulation.py RewardCalculator) ──
LATENCY_MAX, FILE_COUNT_MAX = 15000.0, 2000.0
COMPACT_COST_MAX, PARTITION_COST_MAX, SKEW_MAX = 1.1, 2.0, 2.0
CW_FILES, CW_UTIL, CW_COST, CW_TARGET = 0.25, 0.20, 0.10, 0.45
PW_PRUNING, PW_DELTA, PW_SKEW, PW_COST = 0.55, 0.25, 0.10, 0.10
GW_LATENCY, GW_FILES, GW_PRUNING, GW_UTIL = 0.30, 0.20, 0.30, 0.20
ACTION_COSTS = {0: 0.0, 1: 0.9, 2: 1.0, 3: 1.1, 4: 2.0, 5: 2.0, 6: 2.0, 7: 0.7}

GAMMA = 0.99

def set_seeds(seed):
    import random as _r
    _r.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)


In [ ]:
# ── Cell 2: data discovery (per-episode files only, no eval_* leakage) ──
DATA_GLOBS = ['/kaggle/input/**/*_ep*.csv', './kaggle_training_data/**/*_ep*.csv']

files = []
for p in DATA_GLOBS:
    files.extend(glob.glob(p, recursive=True))
# Per-episode transition files only; exclude anything agent-driven so the
# training pool is purely heuristic-generated (matches the paper's claim):
# eval_* (RL evaluations), MultiAgent, CompactOnly, PartitionOnly (RL ablations)
EXCLUDE = ('eval_', 'MultiAgent', 'CompactOnly', 'PartitionOnly')
files = sorted(set(
    f for f in files
    if 'transitions' in Path(f).name and not any(x in f for x in EXCLUDE)
))
assert files, "No transition CSVs found — attach the kaggle_training_data dataset."

# Two pools, matching the ORIGINAL published training exactly (verified against
# the saved outputs of the original notebooks):
#   • compaction pool: the 14 baseline heuristic episodes ("Total Raw Rows: 14000")
#   • partition pool:  the 6 partition-source episodes (2 runs each of
#     WorkloadAwareThreshold / PartitionExploration / Random_Partition;
#     "6 CSV files ... 6000 total transitions ... 4990 valid partition transitions")
PARTITION_SOURCES = ('WorkloadAwareThreshold', 'PartitionExploration', 'Random_Partition')
files_p = [f for f in files if any(s in f for s in PARTITION_SOURCES)]
files_c = [f for f in files if f not in files_p]
print(f"compaction pool: {len(files_c)} files | partition pool: {len(files_p)} files")
assert len(files_c) == 14 and len(files_p) == 6, "pool sizes differ from the published training"

def load_pool(pool):
    frames = []
    for i, f in enumerate(pool):
        d = pd.read_csv(f)
        d['source_file'] = i      # one file == one episode → windowing boundary
        frames.append(d)
    out = pd.concat(frames, ignore_index=True)
    return out.sort_values(['source_file', 'step']).reset_index(drop=True)

raw_c = load_pool(files_c)
raw_p = load_pool(files_p)
print(f"compaction pool: {len(raw_c)} rows | partition pool: {len(raw_p)} rows")


In [ ]:
# ── Cell 3: reward recomputation (published formulas, next-state based) ──
def target_bonus_vec(actions, rows):
    tb = np.zeros(len(actions), dtype=np.float32)
    low, mid, high = rows <= 50, (rows > 50) & (rows < 150), rows >= 150
    for a, l, m, h in [(1, 1.0, 0.15, -0.6), (2, 0.25, 1.0, 0.4), (3, -0.5, 0.15, 1.0)]:
        sel = actions == a
        tb[sel & low], tb[sel & mid], tb[sel & high] = l, m, h
    return tb

def recompute_rewards(d):
    d = d.copy()
    a = d['action'].fillna(0).astype(int).values
    cost = np.vectorize(lambda x: ACTION_COSTS.get(x, 0.0))(a)
    fc   = d.get('next_file_count', d.get('file_count')).astype(float).values
    util = d.get('next_block_utilization', d.get('block_utilization')).astype(float).values
    lat  = d.get('next_latency_ms', d.get('latency_ms')).astype(float).values
    rows = d.get('rows_ingested', pd.Series(np.zeros(len(d)))).astype(float).values
    skew = d.get('file_size_skew_kb', pd.Series(np.zeros(len(d)))).astype(float).values
    pr_c = d.get('partition_pruning_ratio', pd.Series(np.zeros(len(d)))).astype(float).values
    pr_n = d.get('next_partition_pruning_ratio', pd.Series(pr_c)).astype(float).values

    f_n = np.clip(fc / FILE_COUNT_MAX, 0, 1); u_n = np.clip(util, 0, 1)
    l_n = np.clip(lat / LATENCY_MAX, 0, 1);   s_n = np.clip(skew / SKEW_MAX, 0, 1)
    p_n = np.clip(pr_n, 0, 1);                dp  = np.clip(pr_n - pr_c, -1, 1)

    d['compact_reward'] = (-CW_FILES * f_n + CW_UTIL * u_n
                           - CW_COST * cost / COMPACT_COST_MAX
                           + CW_TARGET * target_bonus_vec(a, rows))
    d['partition_reward'] = (PW_PRUNING * p_n + PW_DELTA * dp
                             - PW_SKEW * s_n - PW_COST * cost / PARTITION_COST_MAX)
    d['global_reward'] = (-GW_LATENCY * l_n - GW_FILES * f_n
                          + GW_PRUNING * p_n + GW_UTIL * u_n)
    return d

df_c = recompute_rewards(raw_c)   # compaction pool (14 baseline episodes)
df_pf = recompute_rewards(raw_p)  # partition pool (6 partition-source episodes)
print("compaction pool rewards:",
      df_c[['compact_reward', 'global_reward']].describe().loc[['mean', 'std']].to_dict())
print("partition pool rewards:",
      df_pf[['partition_reward', 'global_reward']].describe().loc[['mean', 'std']].to_dict())


In [ ]:
# ── Cell 4: features + windows (episode boundaries = source files) ──
def extract_compact_features(d):
    f = np.zeros((len(d), 8), dtype=np.float32)
    f[:, 0] = (d['rows_ingested'].values - ROWS_MEAN) / ROWS_STD
    f[:, 1] = (d['ingestion_rate_rows_per_sec'].values - RATE_MEAN) / RATE_STD
    f[:, 2] = np.clip(d['latency_ms'].values / 15000.0, 0, 1)
    f[:, 3] = np.clip(d['file_count'].values / 2000.0, 0, 1)
    f[:, 4] = np.clip(d['block_utilization'].values, 0, 1)
    f[:, 5] = np.clip(d['total_size_kb'].values / 50000.0, 0, 1)
    f[:, 6] = np.clip(d['file_size_skew_kb'].values / 50.0, 0, 1)
    f[:, 7] = np.exp(-d['steps_since_compact'].values / 10.0)
    return f

def extract_partition_features(d):
    f = np.zeros((len(d), 14), dtype=np.float32)
    f[:, 0] = np.clip(d['latency_ms'].values / 15000.0, 0, 1)
    f[:, 1] = np.clip(d['file_count'].values / 2000.0, 0, 1)
    ps = d['partition_strategy'].values.astype(int)
    for i in range(4):
        f[:, 2 + i] = (ps == i).astype(np.float32)
    f[:, 6] = np.clip(d['partition_pruning_ratio'].values, 0, 1)
    f[:, 7] = np.clip(d['avg_pruning_ratio'].values, 0, 1)
    for j, c in enumerate(['query_hist_time_range', 'query_hist_region_filter',
                           'query_hist_sensor_lookup', 'query_hist_type_filter',
                           'query_hist_full_scan']):
        f[:, 8 + j] = d[c].values
    f[:, 13] = np.exp(-d['steps_since_partition_change'].values / 30.0)
    return f

def build_windows(features, episode_ids, window):
    N, D = features.shape
    out = np.zeros((N, window, D), dtype=np.float32)
    for ep in np.unique(episode_ids):
        idx = np.where(episode_ids == ep)[0]
        for j, i in enumerate(idx):
            s = max(0, j - window + 1)
            seq = features[idx[s:j + 1]]
            out[i, window - len(seq):, :] = seq
    return out

# Compaction: window 10, actions 0-3, discounted compact_reward returns per episode
# (pool = 14 baseline episodes, as in the published training)
ep_c = df_c['source_file'].values
W_c = build_windows(extract_compact_features(df_c), ep_c, 10)
returns_c = np.zeros(len(df_c), dtype=np.float32)
rew_c_full = df_c['compact_reward'].values.astype(np.float32)
for ep in np.unique(ep_c):
    idx = np.where(ep_c == ep)[0]
    G = 0.0
    for i in idx[::-1]:
        G = rew_c_full[i] + GAMMA * G
        returns_c[i] = G

mask_c = df_c['action'].isin([0, 1, 2, 3]).values
Xc, Ac, Gc = W_c[mask_c], df_c['action'].values[mask_c].astype(np.int32), returns_c[mask_c]
Rc = rew_c_full[mask_c]
epc = ep_c[mask_c]
print(f"compaction: {len(Xc)} transitions (published training had 14000)")

# Partition: window 20, actions {0,4,5,6,7} → local 0-4
# (pool = 6 partition-source episodes, as in the published training)
ep_p = df_pf['source_file'].values
W_p = build_windows(extract_partition_features(df_pf), ep_p, 20)
PA_MAP = {0: 0, 4: 1, 5: 2, 6: 3, 7: 4}
mask_p = df_pf['action'].isin(PA_MAP.keys()).values
df_p = df_pf[mask_p].reset_index(drop=True)
Xp = W_p[mask_p]
Ap = np.array([PA_MAP[a] for a in df_p['action'].values], dtype=np.int32)
epp = ep_p[mask_p]
print(f"partition: {len(Xp)} transitions (published training had 4990)")
print("partition action counts:", np.bincount(Ap, minlength=5))


In [ ]:
# ── Cell 5: partition shaped reward (published recipe, replicated exactly) ──
lat_b = df_p['latency_ms'].values.astype(np.float32)
lat_a = df_p['next_latency_ms'].values.astype(np.float32)
lat_improvement = np.clip((lat_b - lat_a) / (lat_b + 1.0), -1, 1)

pr_b = df_p['partition_pruning_ratio'].values.astype(np.float32)
pr_a = df_p['next_partition_pruning_ratio'].values.astype(np.float32)
prune_improvement = np.clip(pr_a - pr_b, -1, 1)

qh = np.column_stack([df_p['query_hist_time_range'], df_p['query_hist_region_filter'],
                      df_p['query_hist_sensor_lookup'], df_p['query_hist_type_filter']]).astype(np.float32)
dominant_query, dominant_strength = np.argmax(qh, axis=-1), np.max(qh, axis=-1)

ideal_action = np.zeros(len(Ap), dtype=np.int32)
ideal_action[dominant_query == 0] = 1   # time_heavy   → HOUR
ideal_action[dominant_query == 1] = 2   # region_heavy → REGION
ideal_action[dominant_query == 3] = 3   # type_heavy   → EVENT_TYPE

current_ps = df_p['partition_strategy'].values.astype(int)
already_correct = (current_ps == ideal_action) & (ideal_action > 0)

alignment = np.zeros(len(Ap), dtype=np.float32)
ok = already_correct
alignment[ok & (Ap == 0)] = 0.5
alignment[ok & (Ap == 4)] = -0.8
for a in [1, 2, 3]:
    alignment[ok & (Ap == a) & (Ap != ideal_action)] = -0.5
bad = ~already_correct
for a in [1, 2, 3]:
    alignment[bad & (Ap == a) & (ideal_action == a)] = 1.0
alignment[bad & (Ap == 0)] = -0.3
alignment[bad & (Ap == 4) & (current_ps > 0)] = 0.3
alignment *= np.clip(dominant_strength * 2.5 - 0.5, 0.3, 1.5)

custom_rewards = (0.3 * (0.5 * lat_improvement + 0.5 * prune_improvement)
                  + 0.5 * alignment
                  + 0.2 * df_p['global_reward'].values.astype(np.float32)).astype(np.float32)

adv_p = custom_rewards.copy()
for a_idx in range(5):
    m = Ap == a_idx
    if m.sum() > 1:
        adv_p[m] = (adv_p[m] - adv_p[m].mean()) / (adv_p[m].std() + 1e-8)
print("partition shaped rewards ready. mean:", custom_rewards.mean())


In [ ]:
# ── Cell 6: model classes — EXACT copies of src/agents/model_registry.py ──
class AttentivePPOModel(tf.keras.Model):
    def __init__(self, num_actions, num_features, window_size=10, embed_dim=64, num_heads=4):
        super().__init__()
        self.num_actions, self.window_size = num_actions, window_size
        self.embedding = tf.keras.layers.Dense(embed_dim, activation='relu', name='embedding')
        self.pos_encoding = self.add_weight(name='pos_encoding', shape=(1, window_size, embed_dim),
                                            initializer='glorot_uniform', trainable=True)
        self.attention = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim,
                                                            dropout=0.1, name='self_attention')
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name='norm1')
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name='norm2')
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(embed_dim * 2, activation='relu', name='ffn_dense1'),
            tf.keras.layers.Dropout(0.1),
            tf.keras.layers.Dense(embed_dim, name='ffn_dense2')], name='ffn')
        self.gap = tf.keras.layers.GlobalAveragePooling1D(name='gap')
        self.actor = tf.keras.layers.Dense(num_actions, activation='softmax', name='actor')
        self.critic = tf.keras.layers.Dense(1, name='critic')

    def call(self, x, training=False):
        e = self.embedding(x) + self.pos_encoding
        x1 = self.norm1(e + self.attention(e, e, training=training))
        x2 = self.norm2(x1 + self.ffn(x1, training=training))
        ctx = self.gap(x2)
        return self.actor(ctx), self.critic(ctx)


class DuelingDDQNModel(tf.keras.Model):
    def __init__(self, num_actions, num_features, window_size=10, **_):
        super().__init__()
        self.num_actions = num_actions
        self.flatten = tf.keras.layers.Flatten()
        self.shared_fc1 = tf.keras.layers.Dense(256, activation='relu', name='shared_fc1')
        self.shared_bn1 = tf.keras.layers.BatchNormalization(name='shared_bn1')
        self.shared_fc2 = tf.keras.layers.Dense(128, activation='relu', name='shared_fc2')
        self.shared_bn2 = tf.keras.layers.BatchNormalization(name='shared_bn2')
        self.value_fc = tf.keras.layers.Dense(64, activation='relu', name='value_fc')
        self.value_out = tf.keras.layers.Dense(1, name='value_out')
        self.advantage_fc = tf.keras.layers.Dense(64, activation='relu', name='advantage_fc')
        self.advantage_out = tf.keras.layers.Dense(num_actions, name='advantage_out')

    def q_values(self, x, training=False):
        h = self.shared_bn1(self.shared_fc1(self.flatten(x)), training=training)
        h = self.shared_bn2(self.shared_fc2(h), training=training)
        v = self.value_out(self.value_fc(h))
        a = self.advantage_out(self.advantage_fc(h))
        return v + (a - tf.reduce_mean(a, axis=-1, keepdims=True))

    def call(self, x, training=False):
        q = self.q_values(x, training=training)
        return tf.nn.softmax(q, axis=-1), tf.reduce_max(q, axis=-1, keepdims=True)


class MlpPPOModel(tf.keras.Model):
    def __init__(self, num_actions, num_features, window_size=10, **_):
        super().__init__()
        self.flatten = tf.keras.layers.Flatten()
        self.shared_fc1 = tf.keras.layers.Dense(256, activation='relu', name='shared_fc1')
        self.shared_fc2 = tf.keras.layers.Dense(128, activation='relu', name='shared_fc2')
        self.shared_dropout = tf.keras.layers.Dropout(0.1)
        self.actor_fc = tf.keras.layers.Dense(64, activation='relu', name='actor_fc')
        self.actor_out = tf.keras.layers.Dense(num_actions, activation='softmax', name='actor_out')
        self.critic_fc = tf.keras.layers.Dense(64, activation='relu', name='critic_fc')
        self.critic_out = tf.keras.layers.Dense(1, name='critic_out')

    def call(self, x, training=False):
        h = self.shared_dropout(self.shared_fc2(self.shared_fc1(self.flatten(x))), training=training)
        return self.actor_out(self.actor_fc(h)), self.critic_out(self.critic_fc(h))


def build_model(arch, num_actions, num_features, window_size):
    cls = {'attentive_ppo': AttentivePPOModel, 'ddqn': DuelingDDQNModel, 'mlp_ppo': MlpPPOModel}[arch]
    m = cls(num_actions=num_actions, num_features=num_features, window_size=window_size)
    m(tf.zeros((1, window_size, num_features)))
    return m


In [ ]:
# ── Cell 7: trainers ──
# Compaction (PPO family): AWR on discounted compact_reward returns — published recipe
# (BETA 1.0, LR 3e-4, 150 epochs, batch 64, weight clip 20, critic on returns).
def train_awr_compaction(model, X, A, G, seed, epochs=150, lr=3e-4, beta=1.0, batch=64):
    opt = tf.keras.optimizers.Adam(lr)
    n = len(X); hist = []
    idx_all = np.random.permutation(n)
    split = int(0.85 * n)
    tr, va = idx_all[:split], idx_all[split:]

    @tf.function
    def step(xb, ab, gb):
        with tf.GradientTape() as tape:
            probs, values = model(xb, training=True)
            values = tf.squeeze(values, -1)
            critic = tf.reduce_mean(tf.square(values - gb))
            adv = gb - tf.stop_gradient(values)
            w = tf.minimum(tf.exp(adv / beta), 20.0)
            lp = tf.reduce_sum(tf.math.log(probs + 1e-8) * tf.one_hot(ab, model.num_actions if hasattr(model,'num_actions') else probs.shape[-1]), axis=-1)
            actor = -tf.reduce_mean(w * lp)
            loss = actor + 0.5 * critic
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        order = np.random.permutation(len(tr))
        tot, nb = 0.0, 0
        for s in range(0, len(tr), batch):
            bi = tr[order[s:s + batch]]
            tot += float(step(tf.constant(X[bi]), tf.constant(A[bi]), tf.constant(G[bi]))); nb += 1
        _, vv = model(tf.constant(X[va]), training=False)
        val = float(tf.reduce_mean(tf.square(tf.squeeze(vv, -1) - G[va])))
        hist.append({'epoch': ep + 1, 'loss': tot / nb, 'val_critic': val})
    return hist


# Partition (PPO family): AWR with per-action standardized advantages, balanced
# sampling, entropy bonus 0.10, critic on immediate shaped reward — published recipe.
def train_awr_partition(model, X, A, ADV, R, seed, epochs=200, lr=3e-4, beta=1.0, batch=64, ent=0.10):
    opt = tf.keras.optimizers.Adam(lr)
    n_act = 5
    a_idx = {a: np.where(A == a)[0] for a in range(n_act)}
    spa = min(len(v) for v in a_idx.values())
    hist = []

    @tf.function
    def step(xb, ab, advb, rb):
        with tf.GradientTape() as tape:
            probs, values = model(xb, training=True)
            w = tf.minimum(tf.exp(advb / beta), 20.0)
            logp = tf.math.log(probs + 1e-8)
            lp = tf.reduce_sum(logp * tf.one_hot(ab, n_act), axis=-1)
            actor = -tf.reduce_mean(w * lp)
            entropy = tf.reduce_mean(-tf.reduce_sum(probs * logp, axis=-1))
            critic = tf.reduce_mean(tf.square(tf.squeeze(values, -1) - rb))
            loss = actor + 0.5 * critic - ent * entropy
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        bal = np.concatenate([np.random.choice(a_idx[a], spa, replace=True) for a in range(n_act)])
        np.random.shuffle(bal)
        tot, nb = 0.0, 0
        for s in range(0, len(bal), batch):
            bi = bal[s:s + batch]
            tot += float(step(tf.constant(X[bi]), tf.constant(A[bi]),
                              tf.constant(ADV[bi]), tf.constant(R[bi]))); nb += 1
        hist.append({'epoch': ep + 1, 'loss': tot / nb})
    return hist


# MLP-PPO (both specialists): offline PPO-clip — the published MLP recipe
# (omar-s-data.ipynb cells 8 & 26). old_logp is snapshotted ONCE from the
# initial network; ratio-clipped surrogate on standardized advantages,
# critic on targets R, entropy 0.01, batch 128, grad clip 0.5, LR 3e-4.
def train_ppoclip(model, X, A, ADV, R, seed, epochs=150, lr=3e-4, batch=128,
                  clip_ratio=0.2, value_coef=0.5, ent=0.01):
    opt = tf.keras.optimizers.Adam(lr)
    n = len(X)
    idx_all = np.random.permutation(n)
    split = int(0.85 * n)
    tr, va = idx_all[:split], idx_all[split:]
    n_act = model(tf.constant(X[:1]))[0].shape[-1]

    init_probs, _ = model(tf.constant(X[tr]), training=False)
    old_logp = tf.math.log(tf.reduce_sum(
        init_probs * tf.one_hot(A[tr], n_act), axis=-1) + 1e-8).numpy()

    Xt, At = X[tr], A[tr]
    ADVt, Rt = ADV[tr], R[tr]
    hist = []

    @tf.function
    def step(xb, ab, advb, rb, olb):
        with tf.GradientTape() as tape:
            probs, values = model(xb, training=True)
            values = tf.squeeze(values, -1)
            chosen = tf.reduce_sum(probs * tf.one_hot(ab, n_act), axis=-1)
            ratio = tf.exp(tf.math.log(chosen + 1e-8) - olb)
            actor = -tf.reduce_mean(tf.minimum(
                ratio * advb,
                tf.clip_by_value(ratio, 1 - clip_ratio, 1 + clip_ratio) * advb))
            critic = tf.reduce_mean(tf.square(rb - values))
            entropy = -tf.reduce_mean(tf.reduce_sum(probs * tf.math.log(probs + 1e-8), -1))
            loss = actor + value_coef * critic - ent * entropy
        grads = tape.gradient(loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 0.5)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        order = np.random.permutation(len(Xt))
        tot, nb = 0.0, 0
        for s in range(0, len(Xt), batch):
            bi = order[s:s + batch]
            tot += float(step(tf.constant(Xt[bi]), tf.constant(At[bi]),
                              tf.constant(ADVt[bi]), tf.constant(Rt[bi]),
                              tf.constant(old_logp[bi]))); nb += 1
        _, vv = model(tf.constant(X[va]), training=False)
        val = float(tf.reduce_mean(tf.square(tf.squeeze(vv, -1) - R[va])))
        hist.append({'epoch': ep + 1, 'loss': tot / nb, 'val_critic': val})
    return hist


# DDQN (both specialists): Double DQN on RAW dueling Q-values; replay pairs are
# consecutive transitions WITHIN episodes (fixes the shuffled-s' defect of the
# original offline notebook).
def train_ddqn(model, target, X, A, R, EP, seed, epochs=150, lr=1e-4, batch=64, target_freq=50):
    opt = tf.keras.optimizers.Adam(lr)
    S, A2, R2, NS, D = [], [], [], [], []
    for ep in np.unique(EP):
        idx = np.where(EP == ep)[0]
        for j in range(len(idx)):
            i = idx[j]
            if j + 1 < len(idx):
                S.append(X[i]); A2.append(A[i]); R2.append(R[i]); NS.append(X[idx[j + 1]]); D.append(0.0)
            else:
                S.append(X[i]); A2.append(A[i]); R2.append(R[i]); NS.append(np.zeros_like(X[i])); D.append(1.0)
    S, A2, R2 = np.array(S, np.float32), np.array(A2, np.int32), np.array(R2, np.float32)
    NS, D = np.array(NS, np.float32), np.array(D, np.float32)
    n_act = int(A.max()) + 1 if A.max() >= 3 else 4
    n_act = model.num_actions

    @tf.function
    def step(s, a, r, ns, d):
        with tf.GradientTape() as tape:
            q = model.q_values(s, training=True)
            q_sel = tf.reduce_sum(q * tf.one_hot(a, n_act), axis=-1)
            na = tf.argmax(model.q_values(ns, training=False), axis=-1)
            tq = target.q_values(ns, training=False)
            tq_sel = tf.reduce_sum(tq * tf.one_hot(na, n_act), axis=-1)
            tgt = r + GAMMA * tq_sel * (1.0 - d)
            loss = tf.reduce_mean(tf.square(tgt - q_sel))
        grads = tape.gradient(loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    hist, updates = [], 0
    for ep in range(epochs):
        order = np.random.permutation(len(S))
        tot, nb = 0.0, 0
        for s0 in range(0, len(S), batch):
            bi = order[s0:s0 + batch]
            tot += float(step(tf.constant(S[bi]), tf.constant(A2[bi]), tf.constant(R2[bi]),
                              tf.constant(NS[bi]), tf.constant(D[bi])))
            nb += 1; updates += 1
            if updates % target_freq == 0:
                target.set_weights(model.get_weights())
        hist.append({'epoch': ep + 1, 'loss': tot / nb})
    return hist


In [ ]:
# ── Cell 8: main loop — 3 archs × 2 specialists × 5 seeds ──
# Published per-run budgets and recipes, verified against the original notebooks:
#   attentive compaction : AWR on returns,               150 ep, batch 64,  no entropy
#   attentive partition  : AWR shaped + balanced,        200 ep, batch 64,  entropy 0.10
#   mlp compaction       : offline PPO-clip on returns,  150 ep, batch 128, entropy 0.01
#   mlp partition        : offline PPO-clip shaped,      150 ep, batch 128, entropy 0.01 (no balancing)
#   ddqn (both)          : double-DQN,                   150 ep, batch 64,  target-update 50
# NOTE: budgets/algorithms deliberately differ per architecture because that is the
# published configuration; the fully recipe-matched comparison happens in Phase 7.
HYPERPARAMS = {
    'attentive_ppo': {'compaction': dict(epochs=150, lr=3e-4, beta=1.0, batch=64),
                      'partition':  dict(epochs=200, lr=3e-4, beta=1.0, batch=64, ent=0.10)},
    'mlp_ppo':       {'compaction': dict(epochs=150, lr=3e-4, batch=128, clip_ratio=0.2, value_coef=0.5, ent=0.01),
                      'partition':  dict(epochs=150, lr=3e-4, batch=128, clip_ratio=0.2, value_coef=0.5, ent=0.01)},
    'ddqn':          {'compaction': dict(epochs=150, lr=1e-4, batch=64, target_freq=50),
                      'partition':  dict(epochs=150, lr=1e-4, batch=64, target_freq=50)},
}

runs = []
for seed in SEEDS:
    for arch in ARCHS:
        for spec in ['compaction', 'partition']:
            tag = f"{spec}_{arch}_seed{seed}"
            wpath = OUTPUT_DIR / 'weights' / f"{tag}.weights.h5"
            if wpath.exists():
                print(f"⏭️  {tag} exists — skip"); continue
            print(f"\n════ {tag} ════")
            set_seeds(seed)
            hp = HYPERPARAMS[arch][spec]
            t0 = time.time()

            if spec == 'compaction':
                na, nf, ws = 4, 8, 10
                model = build_model(arch, na, nf, ws)
                if arch == 'ddqn':
                    tgt = build_model(arch, na, nf, ws); tgt.set_weights(model.get_weights())
                    hist = train_ddqn(model, tgt, Xc, Ac, Rc, epc, seed, **hp)
                elif arch == 'mlp_ppo':
                    # published MLP recipe: PPO-clip, advantages = z-scored returns
                    adv_c = (Gc - Gc.mean()) / max(Gc.std(), 1e-8)
                    hist = train_ppoclip(model, Xc, Ac, adv_c.astype(np.float32), Gc, seed, **hp)
                else:
                    hist = train_awr_compaction(model, Xc, Ac, Gc, seed, **hp)
                n_samples = len(Xc)
            else:
                na, nf, ws = 5, 14, 20
                model = build_model(arch, na, nf, ws)
                if arch == 'ddqn':
                    tgt = build_model(arch, na, nf, ws); tgt.set_weights(model.get_weights())
                    hist = train_ddqn(model, tgt, Xp, Ap, custom_rewards, epp, seed, **hp)
                elif arch == 'mlp_ppo':
                    # published MLP recipe: PPO-clip on shaped advantages, no balancing
                    hist = train_ppoclip(model, Xp, Ap, adv_p, custom_rewards, seed, **hp)
                else:
                    hist = train_awr_partition(model, Xp, Ap, adv_p, custom_rewards, seed, **hp)
                n_samples = len(Xp)

            wall = time.time() - t0
            model.save_weights(str(wpath))
            pd.DataFrame(hist).to_csv(OUTPUT_DIR / 'curves' / f"{tag}_curve.csv", index=False)
            n_params = int(sum(np.prod(v.shape) for v in model.trainable_variables))
            steps_total = len(hist) * max(1, n_samples // hp.get('batch', 64))
            manifest = {
                'tag': tag, 'seed': seed, 'arch': arch, 'specialist': spec,
                'algorithm': ('double_dqn_rawQ' if arch == 'ddqn'
                              else 'offline_ppo_clip' if arch == 'mlp_ppo'
                              else ('awr_returns' if spec == 'compaction' else 'awr_shaped_balanced')),
                'hyperparams': {**hp, 'gamma': GAMMA,
                                'window_size': ws, 'num_features': nf, 'num_actions': na},
                'normalization': {'rows_mean': ROWS_MEAN, 'rows_std': ROWS_STD,
                                  'rate_mean': RATE_MEAN, 'rate_std': RATE_STD},
                'n_train_samples': int(n_samples),
                'trainable_params': n_params,
                'approx_gradient_updates': int(steps_total),
                'wall_clock_sec': round(wall, 1),
                'final_loss': hist[-1]['loss'],
                'gpu': GPU_NAME, 'platform': platform.platform(),
                'tf_version': tf.__version__,
                'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
            }
            with open(OUTPUT_DIR / 'weights' / f"{tag}_manifest.json", 'w') as f:
                json.dump(manifest, f, indent=2)
            runs.append(manifest)
            print(f"   done in {wall:.0f}s — {n_params:,} params, final loss {hist[-1]['loss']:.4f}")

pd.DataFrame([{k: v for k, v in r.items() if k not in ('hyperparams', 'normalization')}
              for r in runs]).to_csv(OUTPUT_DIR / 'training_summary.csv', index=False)
print(f"\n✅ {len(runs)} runs complete")


In [ ]:
# ── Cell 9: package outputs ──
zpath = '/kaggle/working/phase1_weights.zip' if Path('/kaggle').exists() else './phase1_weights.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.rglob('*')):
        if p.is_file():
            z.write(p, p.relative_to(OUTPUT_DIR.parent))
print("→", zpath)
print("Unpack into: revision/phase1_stats/  (giving revision/phase1_stats/phase1_weights/...)")
